# 00 — Setup & Connect

This notebook verifies the IrisPark environment, installs the Jupyter extras, connects to IRIS, and confirms the seed `vendas` table is available.

**Prerequisites**: Python 3.10+, an InterSystems IRIS instance (2025.3+), and the `vendas` seed table (created by `make setup`).

## 1. Environment check

Confirm Python and IrisPark versions.

In [ ]:
import sys
import irispark

print("Python:", sys.version)
print("IrisPark:", irispark.__version__ if hasattr(irispark, "__version__") else "(no __version__ attr)")

## 2. Install Jupyter extras (if needed)

Run this in a terminal if the `jupyter` extra is not installed:

```bash
pip install -e ".[jupyter]"
```

This installs `jupyter` and `ipykernel`.

## 3. Connect to IRIS

The connection uses environment variables with sensible defaults. If IRIS is unreachable, the cell prints `SKIP` and sets `session = None`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 4. Verify the seed table

The `vendas` table (12 rows: `id, cidade, estado, valor, data`) is created by `make setup`.

In [ ]:
df = session.table("vendas")
df.show()
print("Rows:", df.count())

## 5. Deployment diagnostic

Run `irispark-doctor` from a terminal to check IRIS connection, version, CPU flags, and columnar support:

```bash
irispark-doctor
```

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")